In [ ]:
from abc import ABC, abstractmethod

class CabService(ABC):
    def __init__(self, service_name, base_fare, rate_per_km):
        self.service_name = service_name
        self.base_fare = base_fare
        self.rate_per_km = rate_per_km
    @abstractmethod
    
    def calculate_fare(self, distance, ride_type, peak_hour):
        pass
        
    def __str__(self):
        return self.service_name

class Rapido(CabService):
    def __init__(self):
        super().__init__("Rapido", 30, 15)
        
    def calculate_fare(self, distance, ride_type, peak_hour):
        multipliers = {"Bike": 1.00,"Auto": 1.20,"Cab": 1.50}
        distance_fare = self.rate_per_km * distance
        ride_fare = (self.base_fare + distance_fare) * multipliers[ride_type]
        peak_surcharge = 0
        if peak_hour:
            peak_surcharge = ride_fare * 0.20
        total_fare = ride_fare + peak_surcharge
        return {
            "base_fare": self.base_fare,
            "distance_fare": distance_fare,
            "peak_surcharge": peak_surcharge,
            "total_fare": total_fare
        }

class Uber(CabService):
    def __init__(self):
        super().__init__("Uber", 50, 20)
        
    def calculate_fare(self, distance, ride_type, peak_hour):
        multipliers = {"Bike": 1.00,"Auto": 1.15, "Cab": 1.40}
        distance_fare = self.rate_per_km * distance
        ride_fare = (self.base_fare + distance_fare) * multipliers[ride_type]
        peak_surcharge = 0
        if peak_hour:
            peak_surcharge = ride_fare * 0.20
        total_fare = ride_fare + peak_surcharge
        return {
            "base_fare": self.base_fare,
            "distance_fare": distance_fare,
            "peak_surcharge": peak_surcharge,
            "total_fare": total_fare
        }

class Ola(CabService):

    def __init__(self):
        super().__init__("Ola", 40, 18)

    def calculate_fare(self, distance, ride_type, peak_hour):
        multipliers = {"Bike": 1.00,"Auto": 1.10,"Cab": 1.35}
        distance_fare = self.rate_per_km * distance
        ride_fare = (self.base_fare + distance_fare) * multipliers[ride_type]
        peak_surcharge = 0
        if peak_hour:
            peak_surcharge = ride_fare * 0.20
        total_fare = ride_fare + peak_surcharge
        return {
            "base_fare": self.base_fare,
            "distance_fare": distance_fare,
            "peak_surcharge": peak_surcharge,
            "total_fare": total_fare
        }


class User:

    def __init__(self, name, subscription=False):
        self.name = name
        self.subscription = subscription

    def __str__(self):
        status = "Active" if self.subscription else "Not Active"
        return (
            f"Name: {self.name}\n"
            f"Subscription: {status}"
        )

class Booking:

    booking_counter = 1001

    def __init__(
        self,
        user,
        cab_service,
        pickup,
        destination,
        distance,
        ride_type,
        peak_hour
    ):

        self.booking_id = Booking.booking_counter
        Booking.booking_counter += 1
        self.user = user
        self.cab_service = cab_service
        self.pickup = pickup
        self.destination = destination
        self.distance = distance
        self.ride_type = ride_type
        self.peak_hour = peak_hour

        fare_details = cab_service.calculate_fare(distance,ride_type,peak_hour)

        self.base_fare = fare_details["base_fare"]
        self.distance_fare = fare_details["distance_fare"]
        self.peak_surcharge = fare_details["peak_surcharge"]

        self.original_fare = fare_details["total_fare"]

        if user.subscription:
            self.discount = self.original_fare * 0.30
        else:
            self.discount = 0

        self.final_fare = (self.original_fare - self.discount)

    def __str__(self):
        peak_status = "Yes" if self.peak_hour else "No"
        return f"""
==================================================
                 BOOKING CONFIRMED
==================================================
Booking ID        : {self.booking_id}
Customer          : {self.user.name}

Cab Service       : {self.cab_service}
Ride Type         : {self.ride_type}

Pickup            : {self.pickup}
Destination       : {self.destination}
Distance          : {self.distance:.2f} km
Peak Hour         : {peak_status}

--------------------------------------------------
FARE BREAKDOWN
--------------------------------------------------
Base Fare         : ₹{self.base_fare:.2f}
Distance Fare     : ₹{self.distance_fare:.2f}
Peak Surcharge    : ₹{self.peak_surcharge:.2f}

Original Fare     : ₹{self.original_fare:.2f}
Subscription Disc.: -₹{self.discount:.2f}

--------------------------------------------------
FINAL FARE        : ₹{self.final_fare:.2f}
==================================================
"""

class CabBookingSystem:

    def __init__(self, user):
        self.user = user
        self.cab_services = {
            1: Rapido(),
            2: Uber(),
            3: Ola()
        }
        self.ride_types = {
            1: "Bike",
            2: "Auto",
            3: "Cab"
        }


    def get_distance(self):
        while True:
            try:
                distance = float(input("Enter distance (km): "))
                if distance > 0:
                    return distance
                print("Distance must be greater than 0.")
            except ValueError:
                print("Please enter a valid number.")

    def select_service(self):
        print("\nAvailable Cab Services:")
        print("1. Rapido")
        print("2. Uber")
        print("3. Ola")
        while True:
            try:
                choice = int(input("Choose a cab service: "))
                if choice in self.cab_services:
                    return self.cab_services[choice]
                print("Please choose 1, 2 or 3.")
            except ValueError:
                print("Please enter a valid number.")

    def select_ride_type(self):
        print("\nRide Types:")
        print("1. Bike")
        print("2. Auto")
        print("3. Cab")
        while True:
            try:
                choice = int(input("Choose ride type: "))
                if choice in self.ride_types:
                    return self.ride_types[choice]
                print("Please choose 1, 2 or 3.")
            except ValueError:
                print("Please enter a valid number.")

    def get_peak_hour(self):
        while True:
            choice = input("\nIs this a peak-hour ride? (yes/no): ").lower()
            if choice == "yes":
                return True
            elif choice == "no":
                return False
            print("Please enter yes or no.")

    def compare_fares(
        self,
        distance,
        ride_type,
        peak_hour
    ):

        print("\n")
        print("=" * 70)
        print("                    FARE COMPARISON")
        print("=" * 70)

        cheapest_service = None
        cheapest_fare = float("inf")

        results = []

        for service in self.cab_services.values():

            fare_details = service.calculate_fare(
                distance,
                ride_type,
                peak_hour
            )

            original_fare = fare_details["total_fare"]

            # Apply subscription discount
            if self.user.subscription:
                discount = original_fare * 0.30
            else:
                discount = 0

            final_fare = original_fare - discount

            results.append(
                (
                    service,
                    original_fare,
                    discount,
                    final_fare
                )
            )
            if final_fare < cheapest_fare:
                cheapest_fare = final_fare
                cheapest_service = service
        print(
            f"{'Service':<15}"
            f"{'Original Fare':<18}"
            f"{'Discount':<15}"
            f"{'Final Fare':<15}"
        )
        print("-" * 70)
        for service, original, discount, final in results:
            print(
                f"{str(service):<15}"
                f"₹{original:<17.2f}"
                f"₹{discount:<14.2f}"
                f"₹{final:<14.2f}"
            )

        print("-" * 70)
        print(f"\nCheapest Option : {cheapest_service}")
        print(f"Lowest Fare     : ₹{cheapest_fare:.2f}")
        print("=" * 70)
        
    def book_ride(self):
        print("\n")
        print("=" * 50)
        print("                    BOOK A RIDE")
        print("=" * 50)
        pickup = input("Enter pickup location: ").strip()
        destination = input("Enter destination: ").strip()
        distance = self.get_distance()
        ride_type = self.select_ride_type()
        peak_hour = self.get_peak_hour()
        self.compare_fares(distance,ride_type,peak_hour)
        cab_service = self.select_service()
        booking = Booking(self.user,cab_service,pickup,destination,distance,ride_type,peak_hour)
        print(booking)
        
    def run(self):
        while True:
            print("\n")
            print("=" * 50)
            print("           SMART CAB BOOKING SYSTEM")
            print("=" * 50)
            print("1. Book a Ride")
            print("2. Compare Cab Fares")
            print("3. Exit")
            choice = input("\nEnter your choice: ")
            if choice == "1":
                self.book_ride()
            elif choice == "2":
                print("\nEnter journey details:")
                distance = self.get_distance()
                ride_type = self.select_ride_type()
                peak_hour = self.get_peak_hour()
                self.compare_fares(distance,ride_type,peak_hour)
            elif choice == "3":
                print("\nThank you for using SmartCab!")
                break
            else:
                print("\nInvalid choice. Please try again.")

def main():
    print("=" * 50)
    print("              WELCOME TO SMARTCAB")
    print("=" * 50)
    name = input("\nEnter your name: ").strip()
    while True:
        subscription_input = input("Do you have an active subscription? (yes/no): ").lower()
        if subscription_input == "yes":
            subscription = True
            break
        elif subscription_input == "no":
            subscription = False
            break
        else:
            print("Please enter yes or no.")
        
    user = User(name,subscription)
    system = CabBookingSystem(user)
    system.run()

if __name__ == "__main__":
    main()